# cuMFFIMiner: research runtimes

One run per original research threshold, using **actual absolute minSup**.
Runtime is in milliseconds and includes file loading, parsing, mining, and in-memory results.
No discarded warm-ups; the first run may include CUDA kernel compilation.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Use this checkout when the notebook is opened inside the cuPAMI repository.
repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cupami").is_dir()), None)
if repo is not None:
    sys.path.insert(0, str(repo))

from cupami import cuMFFIMiner


In [ ]:
DATA_DIR = Path("/home/mithun/research/datasets")  # Change if your datasets are elsewhere.
DEVICE = 0

# Actual absolute minSup values: passed directly, without dividing by ten.
# Dataset, source file, minimum supports, transaction repetitions.
BENCHMARKS = [
    ("Retail", "Fuzzy_retail.csv", [20, 30, 50, 70, 100], 1),
    ("Kosarak", "Fuzzy_kosarak.csv", [400, 450, 500, 600, 1000], 1),
    ("T10I4D1M", "Fuzzy_T10I4D100K.csv", [1000, 2000, 3000, 4000, 5000], 10),
    ("PUMSB", "Fuzzy_pumsb.csv", [10000, 11000, 12000, 13000, 20000], 1),
]


In [ ]:
rows = []
for dataset, filename, minsups, repeat in BENCHMARKS:
    for min_sup in minsups:
        miner = cuMFFIMiner(
            DATA_DIR / filename,
            minSup=min_sup,
            supportScale=1,
            membershipScale=10,
            repeat=repeat,
            device=DEVICE,
            inputMode="gpu",
            allocator="fresh",
        )
        miner.mine()
        runtime_ms = miner.getTotalRuntime() * 1000
        rows.append({"dataset": dataset, "minSup": min_sup, "runtime_ms": runtime_ms})
        print(f"{dataset:10s}  minSup={min_sup:6d}  runtime={runtime_ms:.3f} ms", flush=True)
        del miner

runtimes = pd.DataFrame(rows)
runtimes
